# Credit Card Default Prediction

Comparison of four supervised classifiers (Logistic Regression, Linear SVM, RBF-kernel SVM, Random Forest) on the UCI *Default of Credit Card Clients* dataset.
See `README.md` for context, methodology and results.

## 0. Setup

In [ ]:
import warnings

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, auc, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_curve)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC

# Some grid points (small C) do not fully converge; the warnings are not informative here.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Set to True to reproduce the second experiment (model retrained without
# the Sex, Education_level and Marital_status dummies). Age is kept in both runs.
DROP_DEMOGRAPHICS = False

In [ ]:
def prob_err(y_true, y_pred):
    """Misclassification rate for labels in {-1, +1}.

    A prediction is wrong iff y_true * y_pred <= 0, so this equals 1 - accuracy.
    Kept as an independent sanity check on the accuracy computed by scikit-learn.
    """
    return np.mean(y_true * y_pred <= 0)

## 1. Data loading

In [ ]:
# Download the dataset from Kaggle (mirror of the UCI repository)
path = kagglehub.dataset_download("uciml/default-of-credit-card-clients-dataset")
df = pd.read_csv(path + "/UCI_Credit_Card.csv")

# Rename columns to self-explanatory names (PAY_0 = September 2005, ..., PAY_6 = April 2005)
df = df.rename(columns={
    'LIMIT_BAL': 'Credit_limit', 'SEX': 'Sex', 'EDUCATION': 'Education_level',
    'MARRIAGE': 'Marital_status', 'AGE': 'Age',
    'PAY_0': 'Repay_status_sept', 'PAY_2': 'Repay_status_aug', 'PAY_3': 'Repay_status_july',
    'PAY_4': 'Repay_status_june', 'PAY_5': 'Repay_status_may', 'PAY_6': 'Repay_status_april',
    'BILL_AMT1': 'Bill_amt_sept', 'BILL_AMT2': 'Bill_amt_aug', 'BILL_AMT3': 'Bill_amt_july',
    'BILL_AMT4': 'Bill_amt_june', 'BILL_AMT5': 'Bill_amt_may', 'BILL_AMT6': 'Bill_amt_april',
    'PAY_AMT1': 'Pay_amt_sept', 'PAY_AMT2': 'Pay_amt_aug', 'PAY_AMT3': 'Pay_amt_july',
    'PAY_AMT4': 'Pay_amt_june', 'PAY_AMT5': 'Pay_amt_may', 'PAY_AMT6': 'Pay_amt_april',
    'default.payment.next.month': 'Target'
})

## 2. Exploratory data analysis

In [ ]:
print("--- Dataset info ---")
print(df.info())
print("\n--- Descriptive statistics ---")
display(df.describe())

print("\n--- Missing values per column ---")
print(df.isnull().sum())

# Class balance: ~78% non-default vs ~22% default
plt.figure(figsize=(8, 5))
sns.countplot(x='Target', data=df, hue='Target', palette='viridis', legend=False)
plt.title('Class balance (Target)', fontsize=14)
plt.xlabel('0: No default, 1: Default')
plt.show()

# Credit limit distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['Credit_limit'], bins=30, kde=True, color='royalblue')
plt.title('Credit limit distribution', fontsize=14)
plt.xlabel('Amount (NT$)')
plt.show()

# September repayment status (-2/-1/0 = no delay or revolving, k >= 1 = k months late)
plt.figure(figsize=(8, 5))
sns.countplot(x='Repay_status_sept', data=df, hue='Repay_status_sept', palette='magma', legend=False)
plt.title('Repayment status, September', fontsize=14)
plt.xlabel('Status (-1 = paid duly, 1 = one month late, ...)')
plt.show()

# Age distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['Age'], bins=30, kde=True, color='orange')
plt.title('Age distribution', fontsize=14)
plt.show()

# September bill amount; log-scaled y axis because counts span several orders of magnitude
plt.figure(figsize=(8, 5))
sns.histplot(df['Bill_amt_sept'], bins=30, color='teal')
plt.title('Bill amount distribution, September', fontsize=14)
plt.yscale('log')
plt.xlabel('Bill amount (NT$)')
plt.ylabel('Count (log scale)')
plt.show()

# Pearson correlation matrix of all variables
plt.figure(figsize=(18, 14))
correlation = df.corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1, annot_kws={"size": 9})
plt.title('Correlation matrix', fontsize=18)
plt.show()

print("\n--- Correlation with Target ---")
print(correlation['Target'].sort_values(ascending=False).round(2))

## 3. Feature engineering

In [ ]:
# Avg_Utilization: mean monthly bill over the six months divided by the credit limit.
# Negative means (credit balance in the client's favour) are clipped to 0.
# The feature is computed row-wise, so creating it before the train/test split causes no leakage.
bill_cols = ['Bill_amt_sept', 'Bill_amt_aug', 'Bill_amt_july',
             'Bill_amt_june', 'Bill_amt_may', 'Bill_amt_april']
df['Avg_Utilization'] = df[bill_cols].mean(axis=1).clip(lower=0) / df['Credit_limit']

In [ ]:
# Merge undocumented categories into "Others"
df['Education_level'] = df['Education_level'].replace([0, 5, 6], 4)  # 0, 5, 6 -> 4
df['Marital_status'] = df['Marital_status'].replace([0], 3)          # 0 -> 3

# One-hot encode nominal variables (first level dropped to avoid collinearity)
categorical_cols = ['Sex', 'Education_level', 'Marital_status']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# pandas returns boolean dummies; cast them to 0/1
for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)

## 4. Train/test split and scaling

In [ ]:
dropped_features = ['ID', 'Target']
if DROP_DEMOGRAPHICS:
    dropped_features += ['Education_level_2', 'Education_level_3', 'Education_level_4',
                         'Marital_status_2', 'Marital_status_3', 'Sex_2']

X = df.drop(columns=dropped_features)
y = df['Target'].values

# 30% train (9,000 rows) / 70% test (21,000 rows), stratified on the target.
# The small training set keeps the RBF-SVM grid search tractable (kernel SVM training
# scales roughly quadratically in the number of samples); the large test set lowers the
# variance of the reported metrics.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.70, random_state=42, stratify=y)

print(f"Training set: {X_train_raw.shape}")
print(f"Test set:     {X_test_raw.shape}\n")

# Standardise features (required by SVMs and regularised logistic regression).
# The scaler is fitted on the training set only.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

## 5. Model training and hyperparameter tuning

In [ ]:
# class_weight='balanced' reweights errors inversely to class frequency,
# to counter the 78/22 imbalance.
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    "Linear SVM": LinearSVC(random_state=42, dual=False, class_weight='balanced'),
    "Kernel SVM (RBF)": SVC(random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight='balanced'),
}

param_grids = {
    "Logistic Regression": {'C': [0.01, 0.1, 1, 10, 100]},
    "Linear SVM": {'C': [0.001, 0.01, 0.1, 1, 10]},
    "Kernel SVM (RBF)": {'C': [0.1, 1, 10], 'gamma': ['scale', 0.1, 0.01]},
    "Random Forest": {'n_estimators': [50, 100, 200], 'max_depth': [10, 15, 20, None],
                      'min_samples_leaf': [1, 5]},
}

results = []          # one row per model for the summary table
best_rf_model = None  # tuned Random Forest, reused for feature importance
roc_data = {}         # (fpr, tpr, auc) per model

for model_name, model in models.items():
    print(f"\n{'=' * 70}")
    print(f"--- Tuning: {model_name} ---")

    # 5-fold CV grid search, optimising F1 on the positive (default) class
    grid_search = GridSearchCV(
        estimator=model, param_grid=param_grids[model_name],
        cv=5, scoring='f1', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    print(f"Best parameters: {grid_search.best_params_}")

    if model_name == "Random Forest":
        best_rf_model = best_model

    # Evaluate on the held-out test set
    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred, pos_label=1)
    prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_test, y_pred, pos_label=1)

    # Misclassification rate via prob_err: map labels {0, 1} -> {-1, +1}
    y_test_pm = np.where(y_test == 0, -1, y_test)
    y_pred_pm = np.where(y_pred == 0, -1, y_pred)
    error_rate = prob_err(y_test_pm, y_pred_pm)

    results.append({
        'Model': model_name,
        'Accuracy': f"{acc:.2%}",
        'Precision': f"{prec:.2%}",
        'Recall': f"{rec:.2%}",
        'F1-Score': f"{f1:.2%}",
        'Error rate': f"{error_rate * 100:.2f}%",
    })

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No default', 'Default'],
                yticklabels=['No default', 'Default'])
    plt.title(f"Confusion matrix:\n{model_name} (tuned)")
    plt.ylabel('True class')
    plt.xlabel('Predicted class')
    plt.show()

    # Scores for the ROC curve: class probabilities where available,
    # otherwise the SVM decision function (ROC only needs a ranking)
    if hasattr(best_model, "predict_proba"):
        y_scores = best_model.predict_proba(X_test)[:, 1]
    else:
        y_scores = best_model.decision_function(X_test)

    fpr, tpr, _ = roc_curve(y_test, y_scores)
    roc_data[model_name] = (fpr, tpr, auc(fpr, tpr))

## 6. Results and feature importance

In [ ]:
print("\n" + "=" * 80)
print("MODEL PERFORMANCE SUMMARY (test set)")
print("=" * 80)
df_results = pd.DataFrame(results).set_index('Model')
print(df_results.to_string())
print("=" * 80 + "\n")

# Impurity-based (MDI) feature importance of the tuned Random Forest.
# Note: MDI favours continuous / high-cardinality features over binary dummies.
if best_rf_model is not None:
    importances = best_rf_model.feature_importances_
    indices = np.argsort(importances)[::-1]

    plt.figure(figsize=(10, 6))
    plt.bar(range(X_train.shape[1]), importances[indices], color='teal')
    plt.title("Feature importance (tuned Random Forest)", fontsize=14)
    plt.xticks(range(X_train.shape[1]), X.columns[indices], rotation=90)
    plt.tight_layout()
    plt.show()

## 7. ROC curves

In [ ]:
plt.figure(figsize=(10, 7))
colors = ['blue', 'green', 'red', 'orange']

for (model_name, (fpr, tpr, roc_auc)), color in zip(roc_data.items(), colors):
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{model_name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--', label='Random classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False positive rate (1 - specificity)', fontsize=12)
plt.ylabel('True positive rate (recall)', fontsize=12)
plt.title('ROC curves (test set)', fontsize=15)
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.show()